In [1]:
from typing import Union, Iterator
from pathlib import Path
import os
import tempfile

import numpy as np
from magic_timer import MagicTimer
import datasets
from tokenizers import BertWordPieceTokenizer, Regex, normalizers
from tqdm.notebook import tqdm
import time
from transformers import (
    BertTokenizerFast,
)

from token_writer import TokenWriter, load_tokens, DTYPE

VOCAB_SIZE = 32_768  # from Cramming
RUN_DIR = Path("data") / f"run_{time.strftime('%Y%m%d-%H%M%S')}"
TOKENIZER_PATH = RUN_DIR / "tokenizer.json"

In [ ]:
# # Train a tokenizer
# RUN_DIR.mkdir(exist_ok=True, parents=True)
# with MagicTimer() as timer:
#     dataset = datasets.load_dataset(
#         "sradc/chunked-shuffled-wikipedia20220301en-bookcorpusopen",
#         split="train",
#         revision="0e6fada2dd43136e4a3f637da41de2e596aee674",
#     )
# print(f"Loaded dataset in {timer}")
# tokenizer = BertWordPieceTokenizer()
# tokenizer._tokenizer.normalizer = normalizers.Sequence(
#     [
#         normalizers.Replace(Regex("(``|'')"), '"'),
#         normalizers.NFD(),
#         normalizers.Lowercase(),
#         normalizers.StripAccents(),
#         normalizers.Replace(Regex(" {2,}"), " "),
#         normalizers.Replace(Regex(r"[^\x00-\x7F]+"), ""),
#     ]
# )  # Normalizer based on, https://github.com/JonasGeiping/cramming/blob/50bd06a65a4cd4a3dd6ee9ecce1809e1a9085374/cramming/data/tokenizer_preparation.py#L52

# def tokenizer_training_data() -> Iterator[str]:
#     for i in tqdm(
#         range(len(dataset)),
#         desc="Feeding samples to tokenizer",
#     ):
#         yield dataset[i]["text"]


# with MagicTimer() as timer:
#     tokenizer.train_from_iterator(
#         tokenizer_training_data(),
#         vocab_size=VOCAB_SIZE,
#         min_frequency=2,
#     )
# print(f"Tokenizer trained in {timer}.")
# tokenizer.save(str(TOKENIZER_PATH))

In [ ]:
%env TOKENIZERS_PARALLELISM=true
TOKENIZER_PATH = "data/run_20230718-150213/tokenizer.json"
TOKENS_FILE = "bookcorpus_wiki_tokens.bin"
print(f"Overiding TOKENIZER_PATH to load {TOKENIZER_PATH}")

In [4]:
# # Tokenize corpus and stream bytes to file
# tokenizer = BertTokenizerFast(tokenizer_file=str(TOKENIZER_PATH))
# with MagicTimer() as timer:
#     # use unshuffled dataset and shuffle after tokenization
#     dataset = datasets.load_dataset(
#         "sradc/chunked-wikipedia20220301en-bookcorpusopen",
#         split="train",
#         revision="3b060686dc821da895a86ac05198f980894f63fa",
#     )
# print(f"Loaded dataset in {timer}")
# with MagicTimer() as timer, TokenWriter(TOKENS_FILE) as writer:
#     for i in tqdm(
#         range(len(dataset)),
#         desc="Tokenizing",
#     ):
#         tokens = np.array(tokenizer(dataset[i]["text"])["input_ids"], dtype=DTYPE)
#         writer.write(tokens)
# print(f"Tokenized dataset in {timer}.")

env: TOKENIZERS_PARALLELISM=true
Overiding TOKENIZER_PATH to load data/run_20230718-150213/tokenizer.json


Found cached dataset parquet (/Users/sidneyradcliffe/.cache/huggingface/datasets/sradc___parquet/sradc--chunked-wikipedia20220301en-bookcorpusopen-2e9ae21627579891/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)


Loaded dataset in 1.7 seconds


Tokenizing:   0%|          | 0/33536113 [00:00<?, ?it/s]

Tokenized dataset in 3.1 hours.


In [13]:
tokens = load_tokens(TOKENS_FILE)
print(f"Loaded {len(tokens)} tokens.")
# Get an idea of speed of loading tokens
for i in tqdm(range(10_000_000)):
    tok = tokens[i]
# 2854375 tokens/s... not bad

Loaded 5687682679 tokens.


  0%|          | 0/10000000 [00:00<?, ?it/s]

In [18]:
for i in tqdm(range(0, 5687682679, 5687682679//100000)):
    tok = tokens[i]
# ~6500 tokens/s with ~random access

  0%|          | 0/100002 [00:00<?, ?it/s]

In [13]:
assert np.all(
    np.array(list(tokenizer.get_vocab().values())) >= 0
), "make sure no negative ids, because using unsigned ints to save to disk"

# TokenWriter prototyping

In [ ]:
DTYPE = np.uint32


class TokenWriter:
    def __init__(self, path: Union[str, Path]):
        self.path = Path(path)
        if self.path.exists():
            assert self.path.stat().st_size == 0, f"File {self.path} already exists"
        self.file = open(self.path, "ab")

    def write(self, tokens: np.ndarray):
        assert tokens.dtype == DTYPE
        assert tokens.ndim == 1
        self.file.write(tokens.tobytes())

    def close(self):
        self.file.close()

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()


def load_tokens(path: Union[str, Path]) -> np.ndarray:
    # Reading a stream of bytes from a file into a memory mapped array
    file_size = os.path.getsize(path)
    num_elements = file_size // np.dtype("uint32").itemsize
    mmapped_array = np.memmap(
        "ints.bin", dtype="uint32", mode="r", shape=(num_elements,)
    )
    return mmapped_array


with tempfile.NamedTemporaryFile() as tmp, TokenWriter(tmp.name) as writer:
    writer.write(np.arange(10, dtype=DTYPE))
    writer.write(np.arange(10, dtype=DTYPE) + 10)
    writer.write(np.arange(10, dtype=DTYPE) + 20)

tokens = load_tokens("ints.bin")
tokens

In [ ]:
# Writing a stream of bytes to a file
x = np.arange(100).astype(np.uint32)
batch_size = 10
out_file = Path("ints.bin")
assert out_file.exists() == False
for batch_idx in range(0, len(x), batch_size):
    bytes = x[batch_idx : batch_idx + batch_size].tobytes()
    with open(out_file, "ab") as f:
        f.write(bytes)

In [ ]:
# Reading a stream of bytes from a file into a memory mapped array
file_size = os.path.getsize("ints.bin")
num_elements = file_size // np.dtype("uint32").itemsize
mmapped_array = np.memmap("ints.bin", dtype="uint32", mode="r", shape=(num_elements,))

In [ ]:
mmapped_array